# Road-following CarRacing agent — diagnose, fix, train, validate (Colab)

**What this notebook does**
* **Stage A – Diagnose** your existing SAC models (`best_model.zip`, `sac_final.zip`): action statistics, collapse checks, video.
* **Stage B/C – Correct and train** with the pipeline validated in this project: PPO + action-repeat 4, reward scaling (training only), early stop of stuck episodes (training only), checkpoint/resume on Google Drive.
* **Stage D – Validate** old vs new models on **identical unseen track seeds** with pre-declared success criteria.
* **Stage E – Demonstrate** with videos.

**Honest status of the recipe (measured on a 2-core CPU in this project, 1 seed):** PPO 200k steps → ~650 reward / ~75 % coverage; **400k steps → 789 ± 162 reward, 89 % coverage, 1 of 20 laps completed** on 20 unseen tracks. SAC-from-pixels collapsed to a constant action in every CPU run. Nothing here guarantees a lap-completing driver; the notebook measures it and tells you whether the success criteria were met.

**Run the cells in order. Every cell is safe to re-run after a runtime disconnect.** Your existing models are only *read*; all new files go to a separate Drive folder.

## Cell 1 — Mount Drive, define paths, protect your existing files
Expected output: `OK` lines for the two old models (or a clear `MISSING` message) and the new output folder. Nothing is written next to your old models.

In [ ]:
import os, sys, pathlib
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab: set DRIVE_ROOT below to a local folder.")

# ---- EDIT IF YOUR PATHS DIFFER -------------------------------------------------
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/autonomous-car-racing-runs")
OLD_BEST   = DRIVE_ROOT / "best_model" / "best_model.zip"     # read-only
OLD_FINAL  = DRIVE_ROOT / "sac_final.zip"                      # read-only
FRAME_STACK_OLD = 1      # frames stacked (VecFrameStack) when the OLD models were trained; 1 = none
CODE_ZIP_CANDIDATES = [DRIVE_ROOT / "autonomous-car-racing.zip", pathlib.Path("/content/autonomous-car-racing.zip")]
# --------------------------------------------------------------------------------
NEW_ROOT = DRIVE_ROOT / "improved_v2"      # ALL new outputs live here (separate from your old runs)
assert NEW_ROOT.resolve() != DRIVE_ROOT.resolve(), "NEW_ROOT must be a different folder"
NEW_ROOT.mkdir(parents=True, exist_ok=True)
for p in (OLD_BEST, OLD_FINAL):
    print(("OK      " if p.is_file() else "MISSING ") + str(p))
print("New outputs ->", NEW_ROOT)

## Cell 2 — Unpack the project code and install dependencies
Upload `autonomous-car-racing.zip` to Drive (`DRIVE_ROOT`) or to `/content` first. Only missing packages are installed (Colab already has PyTorch). Expected output: `All checks passed.` from the environment check. `CarRacing-v3` is used (`v2` no longer exists in Gymnasium ≥ 1.0).

In [ ]:
import shutil, subprocess, importlib.util
REPO = pathlib.Path("/content/autonomous-car-racing")
if not REPO.exists():
    zips = [z for z in CODE_ZIP_CANDIDATES if z.is_file()]
    if not zips:
        raise FileNotFoundError("Upload autonomous-car-racing.zip to " + " or ".join(map(str, CODE_ZIP_CANDIDATES)))
    shutil.unpack_archive(str(zips[0]), "/content")
assert (REPO / "requirements.txt").is_file(), "unexpected zip layout"
os.chdir(REPO); sys.path.insert(0, str(REPO))

need = [m for m in ("Box2D", "gymnasium", "stable_baselines3", "cv2", "imageio_ffmpeg", "psutil", "pandas", "yaml", "tensorboard")
        if importlib.util.find_spec(m) is None]
if need:
    print("Installing (missing: %s) ..." % need)
    subprocess.run("apt-get -qq install -y swig > /dev/null", shell=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-r", "requirements.txt"], check=True)
!python scripts/setup_environment.py

## Cell 3 — Persist outputs on Drive and prevent two training jobs at once
`models/ checkpoints/ logs/ results/ videos/` inside the repo become symbolic links to `NEW_ROOT`, so checkpoints survive a disconnect. Expected output: five `->` lines.

In [ ]:
for d in ("models", "checkpoints", "logs", "results", "videos"):
    target = NEW_ROOT / d; target.mkdir(exist_ok=True)
    link = REPO / d
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        aside = REPO / (d + "_shipped")      # keep anything the zip shipped (e.g. sample weights/videos), never delete it
        if not aside.exists():
            link.rename(aside)
        else:
            shutil.rmtree(link)
    link.symlink_to(target, target_is_directory=True)
    print(d, "->", target)

LOCK = pathlib.Path("/content/train.lock")     # lives on the runtime disk -> disappears with the runtime (never stale)
def acquire_lock():
    if LOCK.exists():
        pid = int(LOCK.read_text() or 0)
        try:
            os.kill(pid, 0); raise RuntimeError(f"Another training job (pid {pid}) is running. Wait for it or stop it first.")
        except ProcessLookupError:
            pass
    LOCK.write_text(str(os.getpid()))
def release_lock():
    LOCK.unlink(missing_ok=True)

## Cell 4 — Stage A: diagnose the old models (read-only)
Rolls out each old model on **5 fixed unseen tracks** (seeds 50000–50004), deterministic and stochastic, and reports steering / gas / brake statistics plus heuristic verdicts. **Evaluation never modifies the model.**

How to read it: `std-in-episode` ≈ 0 for all three actions means the policy ignores the image (collapse). Gas ≈ 0.5 **and** brake ≈ 0.5 means the affine action mapping is holding the brake. Reaching 1000 steps with negative reward means the car is crawling or idle, not racing.
Reproduction in this project: a *default-style* SB3 SAC on plain `CarRacing-v3` gave reward ≈ −28, gas 0.49, brake 0.50, steering std 0.000 — very close to your −26.8 best checkpoint (your own output decides; these are measurements of *your* files).

In [ ]:
DIAG = NEW_ROOT / "diagnostics"
for name, path in (("old_best", OLD_BEST), ("old_final", OLD_FINAL)):
    if path.is_file():
        print("\n########", name)
        !python -m src.diagnostics --model "{path}" --env legacy --algo sac --frame-stack {FRAME_STACK_OLD} --episodes 5 --both --out "{DIAG}"
    else:
        print("skipping", name, "- file missing")
from IPython.display import Image, display
for png in sorted(DIAG.glob("*.png")):
    print(png.name); display(Image(str(png)))

If loading fails with an observation-space error, your old models were trained with a different wrapper stack. Set `FRAME_STACK_OLD` (Cell 1) to the number of stacked frames you used (and re-run). If you used grayscale/resizing wrappers, tell me their exact settings — those models cannot be loaded into a plain environment.

## Cell 5 — Stage A: video of the old best model
Expected: a short clip you can watch inline (steering, whether it drives straight, stands still, or leaves the road).

In [ ]:
from src.diagnostics import load_legacy_model, record_video
from IPython.display import Video
if OLD_BEST.is_file():
    m, v = load_legacy_model(OLD_BEST, "sac", FRAME_STACK_OLD, render_mode="rgb_array")
    p = record_video(m, v, NEW_ROOT / "videos" / "old_best.mp4", seed=50000, deterministic=True, max_steps=500)
    v.close(); display(Video(str(p), embed=True, width=500))

## Cell 6 — Success criteria (defined **before** training)
These are fixed now and checked mechanically in Cell 10. They are judged on **20 unseen track seeds** with the *same seeds for every model*.

| Criterion | Threshold | Why |
|---|---|---|
| Mean track coverage | ≥ 80 % | proves it follows the road, not just survives 1000 steps |
| Mean native reward | ≥ 700 | a clean lap scores ≈ 900 |
| Laps completed | ≥ 50 % of episodes | “drives around the track reliably” |
| Collapse verdict | none | action statistics must be input-dependent |

For reference, the best result measured in this project (PPO, 400k steps, 1 seed, 20 unseen tracks) was 89 % coverage / 789 reward / 5 % laps: it met the first two criteria and **missed the lap criterion**. If yours doesn't meet all three, the notebook will say so.

In [ ]:
CRITERIA = {"coverage_mean": 0.80, "reward_mean": 700.0, "lap_rate": 0.50}
TEST_SEEDS_START, TEST_EPISODES = 20000, 20         # final comparison (never used for training or checkpoint selection)
SELECT_SEEDS_START, SELECT_EPISODES = 30000, 8      # checkpoint selection only
print(CRITERIA)

## Cell 7 — Stage C: train (PPO profile validated here; resumes automatically)
* `configs/ppo_fast.yaml`: action repeat 4, 4 stacked 64×64 gray frames, `reward_scale 0.1` and *stuck-episode truncation* (training only; evaluation always uses the original `CarRacing-v3` reward), `log_std_init −1`, 2 subprocess envs.
* **Re-running this cell after a disconnect resumes from the latest checkpoint** in `NEW_ROOT/checkpoints/ppo_road`.
* Expected: ~35 steps/s on a 2-core runtime → 400k steps ≈ 3 h (in this project 200k steps gave ~75 % coverage and 400k gave ~89 %; evaluation improved late in training, so do not stop at 200k if you can afford more). The simulator runs on the CPU, so a GPU does not speed this up (the CNN is small; `--device cpu` is used on purpose). Keep the tab open; use `TIMESTEPS = 40000` for a ~20 min dry run.

In [ ]:
RUN, TIMESTEPS = "ppo_road", 400_000
has_ckpt = any((REPO / "checkpoints" / RUN).glob("ckpt_*_steps.zip")) or any((REPO / "checkpoints" / RUN).glob("interrupted_*_steps.zip"))
acquire_lock()
try:
    if has_ckpt:
        print("Resuming", RUN)
        !python -m src.train --resume checkpoints/{RUN} --timesteps {TIMESTEPS} --no-final-eval
    else:
        !python -m src.train --config configs/ppo_fast.yaml --run-name {RUN} --timesteps {TIMESTEPS} --device cpu --no-final-eval
finally:
    release_lock()

## Cell 8 — TensorBoard
Watch `rollout/ep_rew_mean`, `rollout/ep_coverage_mean` (training episodes, native reward), `train/value_loss`, `train/std`, `eval/*`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

## Cell 9 — Choose the checkpoint on separate seeds
Re-scores every checkpoint on 8 unseen tracks (seeds 30000+) with **stochastic actions** (the policy was trained with action noise; in this project the deterministic mean scored 138 vs 524 for the same checkpoint). Copies the winner to `models/ppo_road/best_selected.zip`; `best_model.zip` and the checkpoints are not modified. Expected: one line per checkpoint, then `Best: ...`.

In [ ]:
!python scripts/select_checkpoint.py --run {RUN} --episodes {SELECT_EPISODES} --seed-start {SELECT_SEEDS_START} --stochastic --min-step 160000

## Cell 10 — Stage D: old vs new on identical unseen seeds + success check
Each model plays the **same 20 tracks** (seeds 20000–20019). Includes a random-action baseline. Slow part: the old models (1000 steps/episode). Lower `TEST_EPISODES` for a quick look. Expected: a table and PASS/FAIL per criterion for the new model.

In [ ]:
import pandas as pd
from src.diagnostics import run_comparison
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 20)
seeds = [TEST_SEEDS_START + i for i in range(TEST_EPISODES)]
specs = [{"name": "random", "env": "random"}]
for name, path in (("old_best", OLD_BEST), ("old_final", OLD_FINAL)):
    if path.is_file():
        specs += [{"name": name, "env": "legacy", "algo": "sac", "path": str(path), "frame_stack": FRAME_STACK_OLD, "deterministic": True},
                  {"name": name, "env": "legacy", "algo": "sac", "path": str(path), "frame_stack": FRAME_STACK_OLD, "deterministic": False}]
NEW_PATH = REPO / "models" / RUN / "best_selected.zip"
specs += [{"name": "new_ppo", "env": "project", "path": str(NEW_PATH), "deterministic": False}]
table = run_comparison(specs, seeds, out_dir=NEW_ROOT / "results" / "final_comparison")
print(table.round(2).drop(columns=["verdict"]).to_string(index=False))

eps = pd.read_csv(NEW_ROOT / "results" / "final_comparison" / "comparison_episodes.csv")
new = eps[eps.model == "new_ppo"]
got = {"coverage_mean": new.coverage.mean(), "reward_mean": new.reward.mean(), "lap_rate": new.lap_finished.mean()}
print("\nSUCCESS CRITERIA for new_ppo on", len(new), "unseen tracks")
for k, thr in CRITERIA.items():
    print(f"  {'PASS' if got[k] >= thr else 'FAIL'}  {k}: {got[k]:.2f} (needs >= {thr})")
print("  verdict heuristics:", table[table.model == "new_ppo"].verdict.iloc[0])

## Cell 11 — Stage E: video of the new agent
Expected: a clip with an on-screen HUD (step, reward, coverage). Watch for recovery behaviour: does it get back on the road after leaving it?

In [ ]:
from IPython.display import Video, display
!python -m src.inference --model models/{RUN}/best_selected.zip --stochastic --episodes 1 --seed-start {TEST_SEEDS_START} --video videos/new_agent.mp4
display(Video(str(NEW_ROOT / "videos" / "new_agent.mp4"), embed=True, width=600))

## Cell 12 (optional, experimental) — SAC with countermeasures for feature collapse
`configs/sac_drq.yaml` = LayerNorm head + critic-only encoder + random-shift augmentation + update-to-data ratio 1 + reward scale + stuck truncation. **Not validated**: on CPU every SAC variant tried here collapsed or stayed input-independent after ~12k steps (too few updates to tell). Run it only after the PPO result is safe, never concurrently (the lock enforces this). Check an early checkpoint with the diagnostic (e.g. `!python -m src.diagnostics --model checkpoints/sac_drq_exp/ckpt_40000_steps.zip --env project --episodes 3`): if `std-in-episode` is ~0, stop and don't spend more compute.

In [ ]:
RUN_SAC = "sac_drq_exp"
has = any((REPO / "checkpoints" / RUN_SAC).glob("ckpt_*_steps.zip"))
acquire_lock()
try:
    if has:
        !python -m src.train --resume checkpoints/{RUN_SAC} --timesteps 300000 --no-final-eval
    else:
        !python -m src.train --config configs/sac_drq.yaml --run-name {RUN_SAC} --device auto --no-final-eval
finally:
    release_lock()

## Cell 13 — Resume / load the best checkpoint (reference)
```python
# resume training (Cell 7 does this automatically); to extend the budget:
!python -m src.train --resume checkpoints/ppo_road --timesteps 400000 --no-final-eval

# evaluate any checkpoint or best_selected.zip on chosen tracks (stochastic = how PPO was trained)
!python -m src.evaluate --model models/ppo_road/best_selected.zip --episodes 20 --stochastic --seed-start 20000 --compare-baselines

# load in Python
from src.agent import load_model; from src.config import load_config
cfg = load_config("models/ppo_road/config.yaml")
model = load_model("models/ppo_road/best_selected.zip", cfg)      # evaluation never changes the file
```
SAC runs with `training.save_replay_buffer: true` also store the replay buffer next to checkpoints, so resuming is exact; PPO has no buffer.

## Cell 14 — Troubleshooting
| Symptom | Likely cause (check with Cell 4's diagnostic) | What to try |
|---|---|---|
| Drives straight / constant steering, `std-in-episode ≈ 0` | policy ignores the image (feature collapse, SAC) | use the PPO profile; for SAC use `sac_drq.yaml`, check early checkpoints, keep `train_freq 1` |
| Gas ≈ 0.5 **and** brake ≈ 0.5, car crawls | affine action mapping holds the brake | this pipeline uses `brake = max(a, 0)`; for custom code never map brake with `(a+1)/2` |
| Spins in circles / constant steer bias | policy mean far from the noisy behaviour it was trained with, or stuck on grass | evaluate with **stochastic** actions for PPO; widen stuck truncation (`stuck_patience`), train longer |
| Leaves the track at the first corner and sits on grass | no recovery behaviour learned | more steps (resume to 400k+), larger `stuck_patience` (e.g. 80) so grass recovery is trainable |
| Reward gets worse after an early peak | unstable value targets / over-fitting to few eval episodes | keep `reward_scale 0.1`, lower learning rate, select checkpoints on separate seeds (Cell 9), average several seeds |
| `Another training job ... is running` | the lock protects you from two jobs | stop the other cell (Runtime → Interrupt) and re-run |
| Obs-space mismatch when loading old models | different wrappers than a plain RGB env | set `FRAME_STACK_OLD`; custom grayscale/resize models need their original wrapper |
| Colab disconnect | — | re-run Cells 1–3 then Cell 7: it resumes from `NEW_ROOT/checkpoints` |